# 12.2 每秒多少樣本？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像用放大鏡逐段看聲音：時間框可重疊**

框長決定分析範圍；hop 決定多久移動一次。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/frames.svg")))

**先想一想：**1600個樣本在16kHz和8kHz下時長一樣嗎？

sample rate是每秒取幾個點；同樣N個樣本用不同rate播放，會改變時長與音高。改metadata不等於真正重取樣。

**把直覺寫成數學：**duration=N/sample_rate；奈奎斯特頻率=sample_rate/2。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
n = 1600
for rate in (16000, 8000):
    print(rate, "seconds", n / rate, "Nyquist Hz", rate / 2)

**如何讀結果：**8kHz的FSDD不能只宣告成16kHz；先真正resample或用正確rate作頻譜。

**只改一件事：**只改rate，計算一個440Hz週期需要幾個樣本。
